# Safe Mode Tutorial (Notebook)

Run the baseline benchmark, safe smoke scenario, and inspect SafeSwitch telemetry directly from a notebook environment.

## 1. Baseline agentic run
Ensure `orchid_ranker` is installed with the `agentic` extra and this notebook is executed from the repo root so `PYTHONPATH=src` resolves correctly.

In [ ]:
%%bash
set -euo pipefail
PYTHONPATH=src python benchmarks/run_agentic_ml100k.py \
  --rounds 2 --top-users 40 --top-items 80 --top-k 6 --dim 8 \
  --log-dir runs/tutorial-baseline-notebook --quick

## 2. Safe smoke scenario
This uses the helper script (adaptive-only, 60×120 slice, 5 rounds).

In [ ]:
%%bash
set -euo pipefail
./scripts/run_ml100k_safe_smoke.sh runs/tutorial-safe-notebook

## 3. Inspect SafeSwitch telemetry
The following cell loads the adaptive JSONL log and tabulates the gate metrics per round.

In [ ]:
import json
from pathlib import Path

import pandas as pd

log_path = Path('runs/tutorial-safe-notebook/adaptive.jsonl')
rows = []
for line in log_path.read_text().splitlines():
    obj = json.loads(line)
    if obj.get('type') == 'round_summary':
        gate = obj.get('safe_gate') or {}
        rows.append({
            'round': obj.get('round'),
            'serve_policy': gate.get('serve_policy'),
            'p_used': gate.get('p_used'),
            'p_next': gate.get('p'),
            'uplift_lcb': gate.get('lcb'),
            'accept_lcb': gate.get('acc_lcb'),
        })
pd.DataFrame(rows)


## 4. Plot mix probability vs. rounds
This visualization highlights when the gate allows the student to serve.

In [ ]:
import matplotlib.pyplot as plt

df = pd.DataFrame(rows)
plt.figure(figsize=(6, 3))
plt.plot(df['round'], df['p_next'], marker='o')
plt.title('SafeSwitch mix probability')
plt.xlabel('Round')
plt.ylabel('p (probability of adaptive)')
plt.ylim(0, 1)
plt.grid(True)
plt.show()